# Poker Engine Exploration

Interactive notebook to explore the poker engine capabilities.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.engine import PokerEngine, PokerValidator
import numpy as np

## Initialize Engine

In [ ]:
engine = PokerEngine(num_players=2)
state = engine.reset()

print(f"Observation space size: {engine.get_observation_space_size()}")
print(f"Action space size: {engine.get_action_space_size()}")
print(f"Current player: {state['current_player']}")
print(f"Legal actions: {state['legal_actions']}")

## Play a Random Game

In [ ]:
engine.reset()
steps = 0

while not engine.is_done() and steps < 100:
    legal_actions = engine._get_legal_actions()
    if not legal_actions:
        break
    
    action = np.random.choice(legal_actions)
    state = engine.step(action)
    
    print(f"Step {steps}: Player {state['current_player']} took action {action}")
    steps += 1

rewards = engine.get_rewards()
print(f"\nGame finished in {steps} steps")
print(f"Final rewards: {rewards}")

## Test Validation

In [ ]:
engine.reset()
state = engine.get_state()

print(f"State valid: {PokerValidator.validate_game_state(state)}")
print(f"Observation valid: {PokerValidator.validate_observation(state['observation'])}")

## Analyze Game Statistics

In [ ]:
num_games = 50
game_lengths = []
player_wins = {agent: 0 for agent in engine.agents}

for _ in range(num_games):
    engine.reset()
    steps = 0
    
    while not engine.is_done() and steps < 1000:
        legal_actions = engine._get_legal_actions()
        if not legal_actions:
            break
        action = np.random.choice(legal_actions)
        engine.step(action)
        steps += 1
    
    game_lengths.append(steps)
    
    rewards = engine.get_rewards()
    for agent, reward in rewards.items():
        if reward > 0:
            player_wins[agent] += 1

print(f"Average game length: {np.mean(game_lengths):.1f} steps")
print(f"Std deviation: {np.std(game_lengths):.1f}")
print(f"Min/Max: {min(game_lengths)}/{max(game_lengths)}")
print(f"\nWin distribution: {player_wins}")

## Load and Test Deep CFR

In [ ]:
from src.training import DeepCFR

observation_size = engine.get_observation_space_size()
action_size = engine.get_action_space_size()

deep_cfr = DeepCFR(
    observation_size=observation_size,
    action_size=action_size,
    num_players=2,
    hidden_layers=[128, 128]
)

print(f"Deep CFR initialized")
print(f"Advantage networks: {len(deep_cfr.advantage_networks)}")
print(f"Strategy networks: {len(deep_cfr.strategy_networks)}")

## Test Strategy Generation

In [ ]:
engine.reset()
observation = engine.get_observation()
legal_actions = engine._get_legal_actions()

strategy = deep_cfr.get_strategy(observation, player_idx=0, legal_actions=legal_actions)

print(f"Legal actions: {legal_actions}")
print(f"Strategy: {strategy}")
print(f"Strategy sum: {strategy.sum():.4f}")
print(f"Legal action probabilities: {strategy[legal_actions]}")

In [ ]:
engine.close()